# CAPEX goal-seek only (no methodology changes)

This notebook reuses the existing project-level outputs from `lcoe_solar_analysis.csv` and solves **only** for the CAPEX value that makes modeled LCOE (`lcoe_wacc`) equal to observed auction price (`sale_price_auction`), holding all other inputs fixed.

In [ ]:
import csv
import math
import statistics

INPUT_CSV = 'lcoe_solar_analysis.csv'
OUTPUT_CSV = 'capex_goal_seek_only_results.csv'


In [ ]:
def parse_float(value):
    try:
        if value is None:
            return math.nan
        text = str(value).strip()
        if text == '':
            return math.nan
        return float(text)
    except (TypeError, ValueError):
        return math.nan

rows = []
with open(INPUT_CSV, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for r in reader:
        sale_price = parse_float(r.get('sale_price_auction'))
        lcoe_wacc = parse_float(r.get('lcoe_wacc'))
        cost_of_capital = parse_float(r.get('cost_of_capital'))
        capex_current = parse_float(r.get('capex'))
        discounted_energy = parse_float(r.get('discounted_energy'))
        opex_wacc = parse_float(r.get('opex_wacc'))

        capex_required = math.nan
        capex_reduction_pct = math.nan
        status = 'invalid'

        valid_numbers = all(not math.isnan(x) for x in [sale_price, lcoe_wacc, cost_of_capital, capex_current, discounted_energy, opex_wacc])

        if valid_numbers and discounted_energy > 0 and capex_current > 0:
            # Existing LCOE identity in repo: lcoe_wacc = (capex / discounted_energy) + opex_wacc
            # Holding discounted_energy and opex_wacc fixed (as requested), solve for capex only:
            # sale_price_auction = (capex_required / discounted_energy) + opex_wacc
            capex_required = (sale_price - opex_wacc) * discounted_energy
            capex_reduction_pct = ((capex_current - capex_required) / capex_current) * 100.0

            if capex_required < 0:
                status = 'not_solvable_with_capex_only'
            elif capex_required < capex_current:
                status = 'solvable_with_capex_reduction'
            else:
                status = 'no_reduction_needed_or_capex_too_low_already'

        rows.append({
            'date': r.get('date_x', ''),
            'power_plant_name': r.get('power_plant_name', ''),
            'sale_price_auction': sale_price,
            'lcoe_wacc': lcoe_wacc,
            'cost_of_capital': cost_of_capital,
            'capex_current': capex_current,
            'capex_required_to_match_auction': capex_required,
            'capex_reduction_pct_required': capex_reduction_pct,
            'capex_status': status,
        })

fieldnames = [
    'date',
    'power_plant_name',
    'sale_price_auction',
    'lcoe_wacc',
    'cost_of_capital',
    'capex_current',
    'capex_required_to_match_auction',
    'capex_reduction_pct_required',
    'capex_status',
]

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f'Wrote {len(rows)} rows to {OUTPUT_CSV}')


In [ ]:
total_projects = len(rows)
solvable = [r for r in rows if r['capex_status'] == 'solvable_with_capex_reduction']
not_solvable = [r for r in rows if r['capex_status'] == 'not_solvable_with_capex_only']

solvable_count = len(solvable)
not_solvable_count = len(not_solvable)

def pct(part, whole):
    return (part / whole * 100.0) if whole else math.nan

solvable_reductions = [r['capex_reduction_pct_required'] for r in solvable if not math.isnan(r['capex_reduction_pct_required'])]

mean_reduction = statistics.mean(solvable_reductions) if solvable_reductions else math.nan
median_reduction = statistics.median(solvable_reductions) if solvable_reductions else math.nan

print('total_projects:', total_projects)
print('solvable_with_capex_reduction:', solvable_count, f'({pct(solvable_count, total_projects):.2f}%)')
print('not_solvable_with_capex_only:', not_solvable_count, f'({pct(not_solvable_count, total_projects):.2f}%)')
print('mean_capex_reduction_pct_required_solvable_only:', f'{mean_reduction:.2f}')
print('median_capex_reduction_pct_required_solvable_only:', f'{median_reduction:.2f}')
